# 02 – Spawn-Rate Half-Life Selection

This notebook evaluates recency half-life candidates and selects one reproducible default for production usage.

Workflow:
1. Build evaluation foundations (data loading and metrics).
2. Run a temporal holdout benchmark across candidates.
3. Select the best candidate using a deterministic rule.
4. Export final results for documentation.

In [1]:
from dataclasses import dataclass
from pathlib import Path
from typing import Optional
import sys

import numpy as np
import pandas as pd
import pm4py

def detect_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for p in candidates:
        if (p / "spawn_rates").exists() and (p / "data").exists():
            return p
    return Path.cwd()

PROJECT_ROOT = detect_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from spawn_rates.rate_table import TZ_NAME, get_holidays, _build_rate_table_from_df

XES_PATH = PROJECT_ROOT / "data" / "BPI Challenge 2017.xes.gz"
TRAIN_SHARE = 0.70
HALF_LIFE_CANDIDATES: list[Optional[int]] = [None, 30, 60, 90, 120]

@dataclass
class EvalResult:
    half_life: str
    days_test: int
    daily_real: float
    daily_model: float
    daily_gap_pct: float
    abs_daily_gap_pct: float
    slot_wmape: float
    mape_14day: float

def load_arrivals(xes_path: Path) -> pd.DataFrame:
    events = pm4py.convert_to_dataframe(pm4py.read_xes(str(xes_path)))
    required = {"case:concept:name", "time:timestamp"}
    missing = required - set(events.columns)
    if missing:
        raise KeyError(f"Missing columns: {sorted(missing)}")

    work = events[["case:concept:name", "time:timestamp"]].copy()
    work["time:timestamp"] = (
        pd.to_datetime(work["time:timestamp"], utc=True, errors="coerce")
        .dt.tz_convert(TZ_NAME)
    )
    work = work.dropna(subset=["time:timestamp"])
    arrivals = (
        work.groupby("case:concept:name", as_index=False)["time:timestamp"]
        .min()
        .sort_values("time:timestamp")
    )
    arrivals["date"] = arrivals["time:timestamp"].dt.floor("D")
    return arrivals

def expected_total(rate_table: dict, dates: pd.DatetimeIndex, holidays_set: set) -> float:
    total = 0.0
    for day in dates:
        wd = int(day.weekday())
        hol = bool(day.date() in holidays_set)
        for hour in range(24):
            total += rate_table.get((wd, hour, hol), 0.0)
    return total

def evaluate(rate_table: dict, arrivals_test: pd.DataFrame, holidays_set: set, name: str) -> EvalResult:
    test = arrivals_test.copy()
    test["date"] = test["time:timestamp"].dt.floor("D")
    test["weekday"] = test["time:timestamp"].dt.weekday
    test["hour"] = test["time:timestamp"].dt.hour
    test["is_holiday"] = test["date"].dt.date.isin(holidays_set)

    test_days = pd.date_range(test["date"].min(), test["date"].max(), freq="D")
    observed_total = float(len(test))
    expected = expected_total(rate_table, test_days, holidays_set)

    daily_real = observed_total / len(test_days)
    daily_model = expected / len(test_days)
    daily_gap_pct = (daily_model / daily_real - 1.0) * 100.0

    observed_slot = (
        test.groupby(["weekday", "hour", "is_holiday"]).size().rename("obs_count").reset_index()
    )

    exposure = pd.DataFrame({"date": test_days})
    exposure["weekday"] = exposure["date"].dt.weekday
    exposure["is_holiday"] = exposure["date"].dt.date.isin(holidays_set)
    exposure = exposure.groupby(["weekday", "is_holiday"]).size().rename("exp_days").reset_index()

    rows = []
    for wd in range(7):
        for hr in range(24):
            for hol in [False, True]:
                rows.append({
                    "weekday": wd,
                    "hour": hr,
                    "is_holiday": hol,
                    "pred_rate": rate_table.get((wd, hr, hol), 0.0),
                })
    pred = pd.DataFrame(rows).merge(exposure, on=["weekday", "is_holiday"], how="left")
    pred["exp_days"] = pred["exp_days"].fillna(0)
    pred["pred_count"] = pred["pred_rate"] * pred["exp_days"]

    cmp_df = pred.merge(observed_slot, on=["weekday", "hour", "is_holiday"], how="left")
    cmp_df["obs_count"] = cmp_df["obs_count"].fillna(0)
    cmp_df["abs_err"] = (cmp_df["pred_count"] - cmp_df["obs_count"]).abs()
    slot_wmape = cmp_df["abs_err"].sum() / max(cmp_df["obs_count"].sum(), 1.0)

    daily_obs = test.groupby("date").size().reindex(test_days, fill_value=0)
    obs_14 = []
    pred_14 = []
    window = 14
    for start in range(0, len(test_days) - window + 1):
        chunk = test_days[start:start + window]
        obs_14.append(float(daily_obs.loc[chunk].sum()))
        pred_14.append(float(expected_total(rate_table, chunk, holidays_set)))
    obs_14 = np.array(obs_14, dtype=float)
    pred_14 = np.array(pred_14, dtype=float)
    mape_14 = float((np.abs(pred_14 - obs_14) / np.maximum(obs_14, 1.0)).mean())

    return EvalResult(
        half_life=name,
        days_test=len(test_days),
        daily_real=daily_real,
        daily_model=daily_model,
        daily_gap_pct=daily_gap_pct,
        abs_daily_gap_pct=abs(daily_gap_pct),
        slot_wmape=float(slot_wmape),
        mape_14day=mape_14,
    )

**Interpretation:** This setup cell defines all reusable components (project-root detection, data loaders, evaluation metrics, and candidate list).
These foundations are necessary for a transparent and reproducible model-selection step.

## Step 1 — Run temporal holdout benchmark across half-life candidates

This cell performs a 70/30 temporal split, evaluates each candidate, and returns a sorted comparison table.

In [2]:
if not XES_PATH.exists():
    raise FileNotFoundError(f"Cannot find log file: {XES_PATH}")

holidays = get_holidays(force_rebuild=False)
holidays_set = {h.date() if hasattr(h, "date") else h for h in holidays}

arrivals = load_arrivals(XES_PATH)
all_days = pd.Series(arrivals["date"].sort_values().unique())
split_idx = int(len(all_days) * TRAIN_SHARE)
if split_idx < 14 or (len(all_days) - split_idx) < 14:
    raise ValueError("Need at least 28 distinct days for 70/30 split.")

train_days = set(all_days.iloc[:split_idx])
test_days = set(all_days.iloc[split_idx:])
arrivals_train = arrivals[arrivals["date"].isin(train_days)].copy()
arrivals_test = arrivals[arrivals["date"].isin(test_days)].copy()

train_df = arrivals_train[["case:concept:name", "time:timestamp"]].copy()

results: list[EvalResult] = []
for half_life in HALF_LIFE_CANDIDATES:
    name = "baseline" if half_life is None else f"hl{half_life}"
    table = _build_rate_table_from_df(
        train_df,
        holidays_set,
        recency_half_life_days=half_life,
    )
    results.append(evaluate(table, arrivals_test, holidays_set, name))

out = pd.DataFrame([r.__dict__ for r in results])
out = out.sort_values(["abs_daily_gap_pct", "slot_wmape", "mape_14day"]).reset_index(drop=True)
print("=== Tiny Half-Life Selection (70/30 temporal split) ===")
out

/Users/lucashi/TUM/Praktikum/Group Exercise/bppso-groupwork/.venv/lib/python3.11/site-packages/pm4py/utils.py:987: UserWarning: In the current version, the import/export operation uses `rustxes` by default for importing/exporting files faster. Please uninstall `rustxes` to revert the behavior.
  warnings.warn("In the current version, the import/export operation uses `rustxes` by default for importing/exporting files faster. Please uninstall `rustxes` to revert the behavior.")


=== Tiny Half-Life Selection (70/30 temporal split) ===


,half_life,days_test,daily_real,daily_model,daily_gap_pct,abs_daily_gap_pct,slot_wmape,mape_14day
0,hl90,110,89.990909,90.214211,0.248138,0.248138,0.141748,0.089622
1,hl120,110,89.990909,89.042796,-1.053566,1.053566,0.139569,0.089173
2,hl60,110,89.990909,92.289446,2.554188,2.554188,0.151847,0.090860
3,baseline,110,89.990909,85.296350,-5.216704,5.216704,0.141815,0.090842
4,hl30,110,89.990909,96.100999,6.789675,6.789675,0.182735,0.097233


**Interpretation:** Focus on `abs_daily_gap_pct` first (volume calibration), then `slot_wmape` and `mape_14day` for structural and horizon consistency.
A strong candidate should keep all three metrics low without overfitting.

### Step 2 — Candidate selection
Sort by aggregate score to identify the best half-life setting for deployment.

In [3]:
best = out.iloc[0]
print("\nSelected default (clean rule: min abs daily gap, then slot_wmape, then mape_14day):")
print(best[["half_life", "abs_daily_gap_pct", "slot_wmape", "mape_14day"]].to_string())

selection_note = pd.DataFrame([
    {
        "selected_half_life": best["half_life"],
        "abs_daily_gap_pct": float(best["abs_daily_gap_pct"]),
        "slot_wmape": float(best["slot_wmape"]),
        "mape_14day": float(best["mape_14day"]),
    }
])
selection_note


Selected default (clean rule: min abs daily gap, then slot_wmape, then mape_14day):
half_life                hl90
abs_daily_gap_pct    0.248138
slot_wmape           0.141748
mape_14day           0.089622


,selected_half_life,abs_daily_gap_pct,slot_wmape,mape_14day
0,hl90,0.248138,0.141748,0.089622


**Interpretation:** The top row is the recommended half-life based on balanced performance across calibration and short-horizon accuracy.

### Step 3 — Export ranking
Persist the full ranking table so report figures and conclusions are reproducible.

In [4]:
output_csv = PROJECT_ROOT / "spawn_rates" / "analysis" / "half_life_selection_results.csv"
output_csv.parent.mkdir(parents=True, exist_ok=True)
out.to_csv(output_csv, index=False)
print(f"Saved: {output_csv}")

Saved: /Users/lucashi/TUM/Praktikum/Group Exercise/bppso-groupwork/spawn_rates/analysis/half_life_selection_results.csv


## Report Outcome

- This notebook is the executable version of the tiny half-life benchmark.
- The selection rule is deterministic and fully reproducible.
- The selected half-life can be used directly as the simulation default.